<a href="https://colab.research.google.com/github/jiffy-jacob/Resume-based-Interview-Preparation-application/blob/main/interview_app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RESUME BASED INTERVIEW PREPARATION APP

**DEVELOPMENT PIPELINE**

             1. Install libraries
                      ↓
             2. Load Resume PDFs
                      ↓
             3. Extract text
                      ↓
             4. Clean text
                      ↓
             5. Chunk documents
                      ↓
             6. Generate MiniLM embeddings
                      ↓
             7. Create FAISS vector store
                      ↓
             8. Build Retrieval System
                      ↓
             9. Question Generation
                      ↓
            10. Answer Evaluation
                      ↓
            11. Save required files
                      │
                      ↓
              GOOGLE DRIVE / GitHub
                      │
                      ↓
                 STREAMLIT
                      │
                      ↓
              Final Interview App

**1. Install Libraries**

In [ ]:
!pip install -q requests==2.32.4

In [ ]:
import requests
print(requests.__version__)

2.32.4


**Import Libraries**

In [ ]:
!pip install -q pypdf langchain sentence-transformers faiss-cpu transformers

import pypdf
import langchain
import sentence_transformers
import faiss
import transformers

print("All libraries imported successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 402.7/402.7 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 64.3 MB/s eta 0:00:00
All libraries imported successfully!


In [ ]:
from pypdf import PdfReader
import os
import re

**Create Folders**

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#create project folders in google drive
import os

project_path = "/content/drive/MyDrive/AI_Interview_Assistant"

folders = [
    "data/resume",
    "data/knowledge_base",
    "preprocessing",
    "embeddings",
    "vector_store"
]

for folder in folders:
    os.makedirs(os.path.join(project_path, folder), exist_ok=True)

print("Project folders created successfully!")

Project folders created successfully!


In [ ]:
#Set project path
PROJECT_PATH = "/content/drive/MyDrive/AI_Interview_Assistant"

RESUME_PATH = os.path.join(PROJECT_PATH, "data/resume")
KNOWLEDGE_BASE_PATH = os.path.join(PROJECT_PATH, "data/knowledge_base")
VECTOR_STORE_PATH = os.path.join(PROJECT_PATH, "vector_store")

print("Project path:", PROJECT_PATH)

Project path: /content/drive/MyDrive/AI_Interview_Assistant


**2. Extract Text from uploaded resume pdf**

In [ ]:
from pypdf import PdfReader
import os

RESUME_PATH = "/content/drive/MyDrive/AI_Interview_Assistant/data/resume"

# Find the uploaded PDF
pdf_files = [f for f in os.listdir(RESUME_PATH) if f.lower().endswith(".pdf")]

if not pdf_files:
    print("No PDF found. Please check your resume folder.")
else:
    pdf_path = os.path.join(RESUME_PATH, pdf_files[0])

    reader = PdfReader(pdf_path)

    resume_text = ""

    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            resume_text += page_text + "\n"

    print("Resume extracted successfully!")
    print("Number of pages:", len(reader.pages))
    print("\nExtracted Resume Text:\n")
    print(resume_text)

Resume extracted successfully!
Number of pages: 1

Extracted Resume Text:

JIFFY JACOB
 AI / ML INTERN | DATA SCIENCE
 +91-8547765901 | Kannur, Kerala | jiffy.jacob1410@gmail.com
 linkedin.com/in/jiffy-jacob | github.com/jiffy-jacob
PROFESSIONAL SUMMARY
Data Scientist with hands-on experience in Python, data preprocessing, EDA, feature engineering,
machine learning, NLP, and model evaluation. Built predictive, recommendation, clustering, and
text-classification projects using real-world datasets.
TECHNICAL SKILLS
Programming: Python
Data & ML: Pandas, NumPy, Scikit-learn, EDA, Feature Engineering, Regression, Classification,
Clustering, Model Evaluation
Deep Learning & NLP: TensorFlow/Keras, Neural Networks, TF-IDF, NLP, Sentiment/Emotion
Classification
Recommendation: Collaborative Filtering, User-Based CF, Item-Based CF, Cosine Similarity
Tools: Matplotlib, Seaborn, Jupyter, Google Colab, Git, GitHub
MACHINE LEARNING PROJECTS
ML-Powered E-Commerce Sales Prediction & Customer Behavior

Here,
- PdfReader opens our resume PDF.
- extract_text() retrieves the text from each page.
- resume_text combines the extracted text into one string.
- Finally, we display the extracted text so we can check its quality.

**3. Clean the Extracted Text**

In [ ]:
import re

# Create a copy of the extracted text
cleaned_resume_text = resume_text

# Replace unusual bullet characters with standard bullets
cleaned_resume_text = cleaned_resume_text.replace("", "•")

# Remove extra spaces within lines
cleaned_resume_text = re.sub(r"[ \t]+", " ", cleaned_resume_text)

# Remove unnecessary blank lines
cleaned_resume_text = re.sub(r"\n\s*\n+", "\n\n", cleaned_resume_text)

# Remove leading and trailing whitespace
cleaned_resume_text = cleaned_resume_text.strip()

print("Resume text cleaned successfully!")
print("\nCleaned Resume Text:\n")
print(cleaned_resume_text)

Resume text cleaned successfully!

Cleaned Resume Text:

JIFFY JACOB
 AI / ML INTERN | DATA SCIENCE
 +91-8547765901 | Kannur, Kerala | jiffy.jacob1410@gmail.com
 linkedin.com/in/jiffy-jacob | github.com/jiffy-jacob
PROFESSIONAL SUMMARY
Data Scientist with hands-on experience in Python, data preprocessing, EDA, feature engineering,
machine learning, NLP, and model evaluation. Built predictive, recommendation, clustering, and
text-classification projects using real-world datasets.
TECHNICAL SKILLS
Programming: Python
Data & ML: Pandas, NumPy, Scikit-learn, EDA, Feature Engineering, Regression, Classification,
Clustering, Model Evaluation
Deep Learning & NLP: TensorFlow/Keras, Neural Networks, TF-IDF, NLP, Sentiment/Emotion
Classification
Recommendation: Collaborative Filtering, User-Based CF, Item-Based CF, Cosine Similarity
Tools: Matplotlib, Seaborn, Jupyter, Google Colab, Git, GitHub
MACHINE LEARNING PROJECTS
ML-Powered E-Commerce Sales Prediction & Customer Behavior Analytics
 • Buil

During cleaning, we
- Replace unusual bullet characters	to make bullet points readable
- Remove extra spaces	to improve text consistency
- Remove unnecessary blank lines	to make the extracted text more organized
-Strip whitespace	to remove spaces at the beginning and end.
- We are not removing stop words, numbers, or technical terms. Words such as Python, Random Forest, 92.85%, and Hit Rate@10 are important for generating personalized interview questions.

In [ ]:
#Save cleaned Text
cleaned_file_path = os.path.join(
    PROJECT_PATH,
    "preprocessing",
    "cleaned_resume.txt"
)

with open(cleaned_file_path, "w", encoding="utf-8") as file:
    file.write(cleaned_resume_text)

print("Cleaned resume saved to:", cleaned_file_path)

Cleaned resume saved to: /content/drive/MyDrive/AI_Interview_Assistant/preprocessing/cleaned_resume.txt


**4. Divide the resume into meaningful sections**

*This will help our system identify which information relates to skills, projects, education, and other areas.*

In [ ]:
import re

# Define the resume section headings
section_headings = [
    "PROFESSIONAL SUMMARY",
    "TECHNICAL SKILLS",
    "MACHINE LEARNING PROJECTS",
    "CERTIFICATION",
    "EDUCATION"
]

# Split the resume using the section headings
pattern = r"(?=(" + "|".join(map(re.escape, section_headings)) + r"))"

parts = re.split(pattern, cleaned_resume_text)

# Store the sections in a dictionary
resume_sections = {}

for i in range(1, len(parts), 2):
    heading = parts[i].strip()
    content = parts[i + 1].strip() if i + 1 < len(parts) else ""
    resume_sections[heading] = content

# Display the extracted sections
for heading, content in resume_sections.items():
    print(f"\n{'=' * 50}")
    print(heading)
    print('=' * 50)
    print(content)


PROFESSIONAL SUMMARY
PROFESSIONAL SUMMARY
Data Scientist with hands-on experience in Python, data preprocessing, EDA, feature engineering,
machine learning, NLP, and model evaluation. Built predictive, recommendation, clustering, and
text-classification projects using real-world datasets.

TECHNICAL SKILLS
TECHNICAL SKILLS
Programming: Python
Data & ML: Pandas, NumPy, Scikit-learn, EDA, Feature Engineering, Regression, Classification,
Clustering, Model Evaluation
Deep Learning & NLP: TensorFlow/Keras, Neural Networks, TF-IDF, NLP, Sentiment/Emotion
Classification
Recommendation: Collaborative Filtering, User-Based CF, Item-Based CF, Cosine Similarity
Tools: Matplotlib, Seaborn, Jupyter, Google Colab, Git, GitHub

MACHINE LEARNING PROJECTS
MACHINE LEARNING PROJECTS
ML-Powered E-Commerce Sales Prediction & Customer Behavior Analytics
 • Built sales prediction models on Olist e-commerce data using preprocessing, EDA, feature
 engineering, and five ML algorithms.
• Random Forest achieved 

Here,
- It identifies headings: Finds the main sections in your resume.
- It splits the text: Separates the content belonging to each section.
- It creates a dictionary: Stores each heading alongside its content.

**5. Preprare the Knowledge base pdfs**

In [ ]:
import os
from pypdf import PdfReader

KNOWLEDGE_BASE_PATH = os.path.join(
    PROJECT_PATH, "data", "knowledge_base"
)

knowledge_documents = []

for filename in os.listdir(KNOWLEDGE_BASE_PATH):
    if filename.lower().endswith(".pdf"):
        pdf_path = os.path.join(KNOWLEDGE_BASE_PATH, filename)

        reader = PdfReader(pdf_path)
        text = ""

        for page in reader.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"

        knowledge_documents.append({
            "source": filename,
            "text": text.strip()
        })

print("Total PDFs loaded:", len(knowledge_documents))

for document in knowledge_documents:
    print(f"\nFile: {document['source']}")
    print("Characters extracted:", len(document["text"]))

Total PDFs loaded: 4

File: hr_behavioral_interview_questions.pdf
Characters extracted: 6326

File: ai_ml_engineer_interview_guide.pdf
Characters extracted: 6668

File: resume_project_deep_dive_guide.pdf
Characters extracted: 6026

File: data_scientist_interview_guide.pdf
Characters extracted: 7159


In [ ]:
!pip install -q langchain-text-splitters

#Split Text into smaller chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create the text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""]
)

# Split each document into chunks
knowledge_chunks = []

for document in knowledge_documents:
    chunks = text_splitter.split_text(document["text"])

    for i, chunk in enumerate(chunks):
        knowledge_chunks.append({
            "source": document["source"],
            "chunk_id": i,
            "text": chunk
        })

print("Total chunks created:", len(knowledge_chunks))

Total chunks created: 65
